# SQL Avanzado para el analisis de negocio

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import pandas as pd

from src.config import resumen_configuracion
from src.db import consultar, get_engine, probar_conexion
from src.historial_entrenamiento import (
    FECHA_CORTE_POR_DEFECTO,
    HORIZONTE_DIAS,
    SQL_FEATURES,
    construir_features,
    crear_snapshot,
    filas_del_snapshot,
    listar_snapshots,
    resumen_snapshot,
)
from src.schema import crear_tablas
from src.vistas import (
    DEFINICIONES_VISTAS,
    consultar_vista,
    crear_vistas,
    listar_vistas,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 12)
pd.set_option("display.width", 120)

print("Raiz del proyecto:", RAIZ)
print()
print(resumen_configuracion())
print()
print(probar_conexion())

tablas = crear_tablas(get_engine())
print("\nTablas en PostgreSQL:")
for nombre in tablas:
    print(f"  - {nombre}")

print("\nVistas de negocio disponibles:")
for nombre in DEFINICIONES_VISTAS:
    print(f"  - {nombre}")

print(f"\nFecha de corte por defecto: {FECHA_CORTE_POR_DEFECTO}")
print(f"Horizonte del target: {HORIZONTE_DIAS} dias")

Raiz del proyecto: c:\Programacion\DataScienceIA\Docente_Dev_Senior_code\Clases_Master_IA\Modulo_2\Unidad_1\Clase_2_persistencia

PostgreSQL : postgres@localhost:5432/ventas_devsenior
MongoDB    : base 'ml_metadata' (URI configurada: si)
Datos CSV  : C:\Programacion\DataScienceIA\Docente_Dev_Senior_code\Clases_Master_IA\Modulo_2\Unidad_1\Clase_2_persistencia\data\raw

PostgreSQL 17.4 on x86_64-windows, compiled by msvc-19.42.34436, 64-bit

Tablas en PostgreSQL:
  - auditoria
  - clientes
  - dataset_entrenamiento
  - historial_entrenamiento
  - predicciones
  - productos
  - ventas

Vistas de negocio disponibles:
  - v_resumen_ventas
  - v_facturacion_mensual
  - v_margen_categoria
  - v_rendimiento_canal
  - v_ranking_clientes

Fecha de corte por defecto: 2025-06-30
Horizonte del target: 90 dias


In [2]:
conteos = consultar(
    '''
    SELECT 'clientes' AS tabla, COUNT(*) AS filas FROM clientes
    UNION ALL
    SELECT 'productos' AS tabla, COUNT(*) AS filas FROM productos
    UNION ALL
    SELECT 'ventas' AS tabla, COUNT(*) AS filas FROM ventas
    '''
)
conteos

,tabla,filas
0,clientes,10000
1,productos,2000
2,ventas,200000


## Agregaciones complejas

Una agregación **no lista filas**: las **resume**. El negocio pregunta cuánto, quién, cuándo, por dónde y con qué margen.

| Tipo | Pregunta de negocio | Pieza SQL | ¿Colapsa filas? | Vista |
|---|---|---|---|---|
| Global | ¿Cuánto vendimos en total? | `COUNT`, `SUM`, `AVG`, `MIN`, `MAX` (sin `GROUP BY`) | Sí: 200.000 → 1 fila | `v_resumen_ventas` |
| Tiempo | ¿Cómo va cada mes? | `DATE_TRUNC` + `GROUP BY` mes | Sí: 1 fila por mes | `v_facturacion_mensual` |
| Canal | ¿Dónde se vende mejor? | `GROUP BY canal` | Sí: 1 fila por canal | `v_rendimiento_canal` |
| Dimensión | ¿Qué categoría deja más margen? | `JOIN productos` + `GROUP BY categoria` | Sí: 1 fila por categoría | `v_margen_categoria` |
| Identidad | ¿Cuántos clientes distintos? | `COUNT(DISTINCT cliente_id)` | Sigue dentro del grupo | las cuatro de arriba |
| Ranking | ¿Quién es el top, global y por país? | `RANK() OVER (PARTITION BY …)` | No: deja al cliente y le pone puesto | `v_ranking_clientes` |

Regla corta: el `JOIN` va **antes** de agregar. Si agrupas primero, pierdes el coste o el país. Las **CTE** (`WITH`) encadenan varias de estas recetas; eso viene con el snapshot de entrenamiento.

HAVING vs WHERE

WHERE ----> No puede usar SUM(total)

HAVING no sirve para filtrar canal = 'web'

CASE WHEN 

In [3]:
consultar(
    '''
    SELECT
        CASE
            WHEN total < 30 THEN 'pequeno'
            WHEN total < 120 THEN 'medio'
            ELSE 'grande'
        END                         AS tipo_ticket,
        COUNT(*)                    AS num_ventas,
        ROUND(AVG(total), 2)        AS ticket_medio,
        SUM(total)                  AS facturacion
    FROM ventas
    GROUP BY 1
    ORDER BY facturacion DESC
    '''
)

,tipo_ticket,num_ventas,ticket_medio,facturacion
0,grande,100576,787.32,79185863.16
1,medio,66388,67.71,4495158.94
2,pequeno,33036,17.27,570393.63


In [4]:
#CTE (WITH)
#¿Cuanto facturó cada mes y qé porcentaje del total aporta?
#La serie mensual que usamos para un informe de evolución

consultar(
    '''
    WITH mensual AS (
        SELECT 
            DATE_TRUNC('month', fecha_venta)::date AS mes,
            SUM(total)                             AS facturacion
        FROM ventas
        GROUP BY 1
    )
    
    SELECT
        mes,
        facturacion,
        ROUND(
            100.0 * facturacion / (SELECT SUM(facturacion) FROM mensual),
            2
        ) AS pct_del_total 
    FROM mensual
    ORDER BY mes
    '''
)

,mes,facturacion,pct_del_total
0,2022-01-01,18335.62,0.02
1,2022-02-01,53336.90,0.06
2,2022-03-01,173263.34,0.21
3,2022-04-01,182477.22,0.22
4,2022-05-01,223247.66,0.26
...,...,...,...
43,2025-08-01,3453631.99,4.10
44,2025-09-01,3237666.32,3.84
45,2025-10-01,3256593.31,3.87
46,2025-11-01,3269637.62,3.88


In [10]:
#¿Cuántos clientes hay en cada país y como se reparte el segmento?
#platino, oro, plata, bronce

consultar(
    """
    SELECT
        pais,
        COUNT(*)                                         AS clientes,
        COUNT(*) FILTER (WHERE segmento = 'platino')     AS platino,
        COUNT(*) FILTER (WHERE segmento = 'oro')         AS oro,
        COUNT(*) FILTER (WHERE segmento = 'plata')       AS plata,
        COUNT(*) FILTER (WHERE segmento = 'bronce')      AS bronce
    FROM clientes
    GROUP BY pais
    ORDER BY clientes DESC
    """
)

,pais,clientes,platino,oro,plata,bronce
0,Espana,2486,126,389,611,1360
1,Colombia,2461,128,375,620,1338
2,Mexico,1678,86,259,409,924
3,Ecuador,857,43,124,221,469
4,Chile,849,39,133,202,475
5,Peru,846,39,135,218,454
6,Argentina,823,39,128,208,448


In [ ]:
#En el canal web, ¿qué categorías facturan más de 400.000?
#WHERE se quedar con las ventas web y HAVING tira las categorias pequeñas

consultar(
    """
    SELECT 
        p.categoria,
        COUNT(*)         AS num_ventas_web,
        SUM(v.total)     AS facturacion_web
    FROM ventas v
    JOIN productos p ON v.producto_id = p.producto_id
    WHERE v.canal = 'web'
    GROUP BY p.categoria
    HAVING SUM(v.total) > 400000
    ORDER BY facturacion_web DESC
    """
)

,categoria,num_ventas_web,facturacion_web
0,Electronica,12601,19853091.45
1,Deportes,12271,7911658.05
2,Hogar,12735,5139039.96
3,Ropa,12256,1903494.59
4,Juguetes,12851,1641776.16
5,Libros,13775,927648.65
6,Alimentacion,13477,559084.16


In [ ]:
#¿Cómo evoluciona la facturación por año y trimestre?

In [ ]:
#¿Qué informes reutilizables hay en PostgreSQL?

In [ ]:
#¿Cuál es el panorama global del negocio?

## WINDOW FUNCTIONS

GROUP BY ----------------> 200.000 -----> 7 categorias

WF -----------> Mantiene 200.000 filas y lo que hace es que añade un acoluman extra

SUM(total) OVER () 


ANATOMIA de un FUNCTION

FUNCION(...) OVER (
    PARTITION BY
    ORDER BY
    ROWS BETWEEN 
)

LAG / LEAD

In [ ]:
# ¿Cuales son los productos que más factura y como se numera el puesto?
consultar(
    """
    SELECT
        p.nombre,
        p.categoria,
        SUM(v.total) AS facturacion,
        ROW_NUMBER() OVER (ORDER BY SUM(v.total) DESC) AS n_fila,
        RANK()       OVER (ORDER BY SUM(v.total) DESC) AS puesto_con_hueco,
        DENSE_RANK() OVER (ORDER BY SUM(v.total) DESC) AS puesto_sin_hueco
    FROM ventas v
    JOIN productos p ON v.producto_id = p.producto_id  
    GROUP BY p.nombre, p.categoria
    ORDER BY facturacion
    LIMIT 12
    """
)

,nombre,categoria,facturacion,n_fila,puesto_con_hueco,puesto_sin_hueco
0,Novela Plus 788,Alimentacion,200.75,2000,2000,2000
1,Mochila Basic 637,Alimentacion,236.47,1999,1999,1999
2,Balon Pro 1707,Alimentacion,296.07,1998,1998,1998
3,Lampara Ultra 95,Alimentacion,338.37,1997,1997,1997
4,Cafetera Mini 1498,Alimentacion,356.65,1996,1996,1996
5,Silla Classic 1292,Alimentacion,356.75,1995,1995,1995
6,Mochila Pro 1294,Alimentacion,358.22,1994,1994,1994
7,Raqueta Ultra 71,Alimentacion,368.28,1993,1993,1993
8,Monitor Pro 1119,Alimentacion,389.65,1992,1992,1992
9,Teclado Pro 890,Alimentacion,393.39,1991,1991,1991


In [15]:
#¿Cuaáles son los 3 productos que más facturan dentro de cada categoria?
consultar(
    """
    SELECT categoria, nombre, facturacion, puesto
    FROM(  SELECT
            p.categoria,
            p.nombre,
            SUM(v.total) AS facturacion,
            ROW_NUMBER() OVER (
                PARTITION BY p.categoria
                ORDER BY SUM(v.total) DESC
            ) AS puesto
        FROM ventas v
        JOIN productos p ON v.producto_id = p.producto_id  
        GROUP BY p.categoria, p.nombre
    ) ranking
    WHERE puesto <= 3
    ORDER BY categoria, puesto
    """
)

,categoria,nombre,facturacion,puesto
0,Alimentacion,Zapatillas Plus 1983,9714.62,1
1,Alimentacion,Raqueta Classic 1639,8892.76,2
2,Alimentacion,Sudadera Classic 1971,8833.99,3
3,Deportes,Manual Max 227,146838.41,1
4,Deportes,Lampara Classic 221,140773.53,2
...,...,...,...,...
16,Libros,Camiseta Pro 1413,14189.01,2
17,Libros,Lampara Ultra 1302,14161.91,3
18,Ropa,Manual Mini 285,33102.21,1
19,Ropa,Balon Pro 301,32919.19,2
